In [5]:
import pandas as pd
import numpy as np


####################################################
# Bond Pricing Function
####################################################

def bond_price(face, coupon_rate, ytm, maturity, freq):
    periods = int(maturity * freq)
    coupon = face * coupon_rate / freq
    r = ytm / freq
    price = 0
    for t in range(1, periods + 1):
        price += coupon / (1+r)**t
    price += face / (1+r)**periods
    return price

####################################################
# Duration Function
####################################################

def modified_duration(face, coupon_rate, ytm, maturity, freq):
    periods = int(maturity * freq)
    coupon = face * coupon_rate / freq
    r = ytm / freq
    price = bond_price(
        face,
        coupon_rate,
        ytm,
        maturity,
        freq)

    weighted_time = 0


    for t in range(1, periods+1):
        cashflow = coupon
        if t == periods:
            cashflow += face
        pv = cashflow/(1+r)**t
        weighted_time += (t/freq)*pv

    macaulay = weighted_time/price
    return macaulay/(1+ytm/freq)

####################################################
# Convexity Function
####################################################

def convexity(face, coupon_rate, ytm, maturity, freq):
    periods = int(maturity*freq)
    coupon = face*coupon_rate/freq
    r = ytm/freq
    price = bond_price(
        face,
        coupon_rate,
        ytm,
        maturity,
        freq)

    conv = 0

    for t in range(1, periods+1):
        cashflow = coupon
        if t == periods:
            cashflow += face
        pv = cashflow/(1+r)**t
        conv += pv*t*(t+1)
    return conv/(price*(1+r)**2*freq**2)

####################################################
# Example Portfolio
####################################################

portfolio = pd.DataFrame({

    "Bond":[
        "US Treasury 5Y",
        "German Bund 10Y",
        "Corporate Bond A",
        "Corporate Bond B" ],


    "FaceValue":[
        1000,
        2000,
        1500,
        3000 ],


    "Coupon":[
        0.035,
        0.020,
        0.050,
        0.045  ],


    # Beginning yield
    "Yield_Start":[
        0.032,
        0.023,
        0.048,
        0.041  ],


    # Ending yield after market movement
    "Yield_End":[
        0.035,
        0.021,
        0.050,
        0.044 ],


    # Credit spread change
    "Spread_Change":[
        0,
        0,
        0.0015,
        0.0020 ],


    "Maturity":[
        5,
        10,
        7,
        12  ],


    "Frequency":[
        2,
        1,
        2,
        2 ]

})



####################################################
# Attribution Calculation
####################################################

results = []


for _, bond in portfolio.iterrows():

    face = bond.FaceValue
    coupon = bond.Coupon
    maturity = bond.Maturity
    freq = bond.Frequency

    ytm_start = bond.Yield_Start
    ytm_end = bond.Yield_End


    # Starting price
    price_start = bond_price(
        face,
        coupon,
        ytm_start,
        maturity,
        freq )


    # Ending price

    price_end = bond_price(
        face,
        coupon,
        ytm_end,
        maturity,
        freq )


    duration = modified_duration(
        face,
        coupon,
        ytm_start,
        maturity,
        freq )


    conv = convexity(
        face,
        coupon,
        ytm_start,
        maturity,
        freq  )

    yield_change = ytm_end - ytm_start

    ################################################
    # Attribution Components
    ################################################

    # 1. Coupon Income
    carry = coupon

    # 2. Duration Effect
    duration_effect = (-duration * yield_change)

    # 3. Convexity Effect
    convexity_effect = (0.5 * conv * yield_change**2 )

    # 4. Credit Spread Effect
    spread_effect = (-duration * bond.Spread_Change )

    # Actual return
    total_return = ((price_end-price_start) / price_start) + carry

    explained_return = (
        carry +
        duration_effect +
        convexity_effect +
        spread_effect )

    residual = ( total_return - explained_return)


    results.append({
        "Bond":bond.Bond,
        "Total Return %": total_return*100,
        "Carry %": carry*100,
        "Duration Effect %": duration_effect*100,
        "Convexity Effect %": convexity_effect*100,
        "Credit Spread Effect %": spread_effect*100,
        "Residual %": residual*100
    })



####################################################
# Output Report
####################################################
attribution = pd.DataFrame(results)
print("\n================ Fixed Income Performance Attribution ================\n")
print(attribution.round(3).T)

####################################################
# Portfolio Attribution Summary
####################################################
print("\n================ Portfolio Contribution ================\n")
print( attribution.sum(numeric_only=True).round(3) )


================ Fixed Income Performance Attribution ================

                                     0                1                 2  \
Bond                    US Treasury 5Y  German Bund 10Y  Corporate Bond A   
Total Return %                   2.143            3.807             3.836   
Carry %                            3.5              2.0               5.0   
Duration Effect %               -1.368            1.789            -1.172   
Convexity Effect %               0.011            0.019             0.008   
Credit Spread Effect %            -0.0             -0.0            -0.879   
Residual %                        -0.0              0.0             0.879   

                                       3  
Bond                    Corporate Bond B  
Total Return %                     1.766  
Carry %                              4.5  
Duration Effect %                  -2.78  
Convexity Effect %                 0.047  
Credit Spread Effect %            -1.854  
Residual 